# Fine-Tuning BERT for Question Answering (QA)
### Hugging Face Learning Series — Step-by-Step Guide

---

## How QA Differs from Classification and NER

| Aspect | Text Classification | NER | Question Answering |
|---|---|---|---|
| Task | One label per sentence | One label per token | Find answer span in context |
| BERT output used | `[CLS]` vector | All token vectors | All token vectors (start + end) |
| Model class | `AutoModelForSequenceClassification` | `AutoModelForTokenClassification` | `AutoModelForQuestionAnswering` |
| Labels | Single integer | Integer per token | Two integers (start_pos, end_pos) |
| Key challenge | None special | Subword alignment | Offset mapping + long context handling |
| Output | POSITIVE / NEGATIVE | B-PER, I-LOC... | "Asansol" (extracted text span) |

---

## How Extractive QA Works

BERT's QA is **extractive** — it does NOT generate an answer.
It finds the answer span that already exists inside the given context.

```
Question : "Where is the Eiffel Tower located?"
Context  : "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France."

BERT predicts:
  start_position = index of token "Paris"
  end_position   = index of token "France"

Answer extracted : "Paris, France"
```

## The QA Head Architecture
```
Input  : [CLS] Question [SEP] Context [SEP]
           ↓
BERT Encoder (12 layers)
           ↓
All token vectors  [seq_len × 768]
           ↓
Linear(768 → 2)   ← Two output logits per token: start_logit and end_logit
           ↓
start_logits [seq_len]    end_logits [seq_len]
           ↓                       ↓
argmax → start_pos        argmax → end_pos
           ↓
tokens[start_pos : end_pos+1] → answer string
```

## The Long Context Challenge
BERT has a max input of 512 tokens. But context paragraphs can be much longer.
The solution is **sliding window** — split the context into overlapping chunks,
run BERT on each chunk, and pick the chunk with the best answer score.

```
Context (800 tokens)
  Chunk 1: tokens   0 → 384  (with 128-token overlap)
  Chunk 2: tokens 256 → 512
  Chunk 3: tokens 384 → 640
  ...
```

---

## What You Will Learn

| Step | What happens |
|---|---|
| 1 | Set up environment & GPU |
| 2 | Authenticate with Hugging Face Hub |
| 3 | Load the SQuAD dataset |
| 4 | Explore the dataset structure |
| 5 | Tokenize with offset mapping & sliding window |
| 6 | Load BERT with QA head |
| 7 | Define training arguments |
| 8 | Define QA evaluation metrics |
| 9 | Train the model |
| 10 | Evaluate with Exact Match & F1 |
| 11 | Run inference on custom questions |
| 12 | Save the model locally |
| 13 | Push to Hugging Face Hub |
| 14 | Load and test the published model |

---
> **Runtime:** `Runtime → Change runtime type → T4 GPU`

---
## Step 1 — Install & Import Libraries

In [1]:
!pip install -q transformers datasets evaluate accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import collections
import evaluate
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
    pipeline
)
from huggingface_hub import notebook_login, HfApi

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device       : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version  : 2.10.0+cu128
CUDA available   : True
GPU device       : Tesla T4
GPU memory       : 15.6 GB


---
## Step 2 — Authenticate with Hugging Face Hub

In [3]:
# Get your write token from: https://huggingface.co/settings/tokens
notebook_login()

In [5]:
# ── Configuration ─────────────────────────────────────────────────────────────

HF_USERNAME    = "argha9177"          # Your Hugging Face username
MODEL_NAME     = "bert-base-uncased"          # Base model to fine-tune
DATASET_NAME   = "rajpurkar/squad"            # SQuAD v1.1 dataset
REPO_NAME      = "bert-squad-qa"             # Your Hub repo name
HF_REPO_ID     = f"{HF_USERNAME}/{REPO_NAME}"

# Tokenization settings
MAX_LENGTH     = 384    # Max total input length (question + context)
DOC_STRIDE     = 128    # Overlap between consecutive sliding window chunks

# Training hyperparameters
BATCH_SIZE     = 16
EPOCHS         = 2      # SQuAD converges fast; 2-3 epochs is standard
LEARNING_RATE  = 3e-5   # Slightly higher than classification is common for QA
WARMUP_RATIO   = 0.1
WEIGHT_DECAY   = 0.01

# Use a subset for faster training in Colab
TRAIN_SUBSET   = 8000
EVAL_SUBSET    = 1000

SAVE_DIR       = "./bert-squad-qa"

print(f"Model      : {MODEL_NAME}")
print(f"Dataset    : {DATASET_NAME}")
print(f"Repo ID    : {HF_REPO_ID}")
print(f"Max length : {MAX_LENGTH}  (question + context tokens)")
print(f"Doc stride : {DOC_STRIDE}  (sliding window overlap)")

Model      : bert-base-uncased
Dataset    : rajpurkar/squad
Repo ID    : argha9177/bert-squad-qa
Max length : 384  (question + context tokens)
Doc stride : 128  (sliding window overlap)


---
## Step 3 — Load the SQuAD Dataset

**SQuAD (Stanford Question Answering Dataset) v1.1** is the standard
benchmark for extractive QA. It contains:
- 87,599 training question-context-answer triples
- 10,570 validation examples
- Questions from Wikipedia passages
- Every answer is a span directly extracted from the context

In [6]:
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print(f"\nTraining examples   : {len(raw_dataset['train'])}")
print(f"Validation examples : {len(raw_dataset['validation'])}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Training examples   : 87599
Validation examples : 10570


In [7]:
# Take subsets for faster Colab training
train_data = raw_dataset['train'].shuffle(seed=42).select(range(TRAIN_SUBSET))
eval_data  = raw_dataset['validation'].shuffle(seed=42).select(range(EVAL_SUBSET))

print(f"Training subset   : {len(train_data)}")
print(f"Evaluation subset : {len(eval_data)}")

Training subset   : 8000
Evaluation subset : 1000


---
## Step 4 — Explore the Dataset Structure

SQuAD's structure is very different from classification or NER.
Each sample has a question, a context paragraph, and an **answers dict**
containing the answer text and its **character-level start position** in the context.

In [8]:
# Dataset features
print("Dataset features:")
print(raw_dataset['train'].features)
print()

Dataset features:
{'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}}



In [9]:
# Inspect 3 examples carefully
print("=" * 70)
print("SAMPLE SQUAD EXAMPLES")
print("=" * 70)

for i in range(3):
    example = train_data[i]
    print(f"\nExample {i+1}:")
    print(f"  ID       : {example['id']}")
    print(f"  Question : {example['question']}")
    print(f"  Context  : {example['context'][:200]}...")
    print(f"  Answers  : {example['answers']}")
    # answers is a dict: {'text': ['answer text'], 'answer_start': [char_index]}
    answer_text  = example['answers']['text'][0]
    answer_start = example['answers']['answer_start'][0]
    print(f"  → Answer text          : '{answer_text}'")
    print(f"  → Character start pos  : {answer_start}")
    print(f"  → Verified in context  : '{example['context'][answer_start:answer_start+len(answer_text)]}'")

SAMPLE SQUAD EXAMPLES

Example 1:
  ID       : 573173d8497a881900248f0c
  Question : What percentage of Egyptians polled support death penalty for those leaving Islam?
  Context  : The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan indepen...
  Answers  : {'text': ['84%'], 'answer_start': [468]}
  → Answer text          : '84%'
  → Character start pos  : 468
  → Verified in context  : '84%'

Example 2:
  ID       : 57277e815951b619008f8b52
  Question : Ann Arbor ranks 1st among what goods sold?
  Context  : The Ann Arbor Hands-On Museum is located in a renovated and expanded historic downtown fire station. Multiple art galleries exist in the city, notably in the downtown area and around the University of...
  Answers  : {'text': ['books'], 'answer_start': [402]}
  → Answer text          : 'books'
  → Character start pos  : 402
  → Verified in con

In [10]:
# Check answer and context length distribution
answer_lengths  = [len(ex['answers']['text'][0].split()) for ex in train_data]
context_lengths = [len(ex['context'].split()) for ex in train_data]

print("Answer length (words):")
print(f"  Mean   : {np.mean(answer_lengths):.1f}")
print(f"  Median : {np.median(answer_lengths):.1f}")
print(f"  Max    : {max(answer_lengths)}")
print()
print("Context length (words):")
print(f"  Mean   : {np.mean(context_lengths):.1f}")
print(f"  Median : {np.median(context_lengths):.1f}")
print(f"  Max    : {max(context_lengths)}")
print(f"  95th % : {np.percentile(context_lengths, 95):.1f}")
print()
print(f"MAX_LENGTH={MAX_LENGTH} with DOC_STRIDE={DOC_STRIDE} handles long contexts via sliding window.")

Answer length (words):
  Mean   : 3.1
  Median : 2.0
  Max    : 29

Context length (words):
  Mean   : 120.1
  Median : 110.0
  Max    : 562
  95th % : 214.0

MAX_LENGTH=384 with DOC_STRIDE=128 handles long contexts via sliding window.


---
## Step 5 — Tokenization with Offset Mapping

QA tokenization is the **most complex** of the three tasks. There are three challenges:

**Challenge 1 — Two inputs, one sequence:**
BERT receives question and context concatenated:
`[CLS] question tokens [SEP] context tokens [SEP]`

**Challenge 2 — Character positions → Token positions:**
SQuAD labels are character offsets. We need to convert them to token indices.
We use `offset_mapping` — each token knows its start and end character position.

**Challenge 3 — Long contexts (sliding window):**
If context > MAX_LENGTH, we create multiple overlapping chunks.
The answer may only be in one of them — we label the others with `(0, 0)`.

### How offset_mapping works:
```
Context : "Paris is the capital"
Tokens  : ['paris', 'is', 'the', 'capital']
Offsets : [(0,5), (6,8), (9,12), (13,20)]
                   ↑
  If answer_start=6 (char), we look for the token
  whose offset contains character 6 → token index 1 → 'is'
```

In [11]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded : {MODEL_NAME}")
print(f"Max model length : {tokenizer.model_max_length}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded : bert-base-uncased
Max model length : 512


In [12]:
# ── Demonstrate offset mapping on one example ────────────────────────────────
sample     = train_data[0]
question   = sample['question']
context    = sample['context']
ans_text   = sample['answers']['text'][0]
ans_start  = sample['answers']['answer_start'][0]

print(f"Question        : {question}")
print(f"Answer text     : '{ans_text}'")
print(f"Answer char pos : {ans_start} → {ans_start + len(ans_text)}")
print()

tokenized = tokenizer(
    question,
    context,
    truncation="only_second",   # Only truncate the context, never the question
    max_length=MAX_LENGTH,
    stride=DOC_STRIDE,
    return_overflowing_tokens=True,  # Create multiple chunks if context is long
    return_offsets_mapping=True,     # Each token maps back to character positions
    padding="max_length"
)

print(f"Number of chunks created : {len(tokenized['input_ids'])}")
print(f"Tokens in chunk 0        : {len(tokenized['input_ids'][0])}")
print()

# Show offset mapping for first few tokens
tokens  = tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])
offsets = tokenized['offset_mapping'][0]
print("First 20 tokens with their character offsets:")
print(f"  {'Token':<20s} {'Char offset'}")
print(f"  {'-'*35}")
for tok, off in zip(tokens[:20], offsets[:20]):
    print(f"  {tok:<20s} {off}")

Question        : What percentage of Egyptians polled support death penalty for those leaving Islam?
Answer text     : '84%'
Answer char pos : 468 → 471

Number of chunks created : 1
Tokens in chunk 0        : 384

First 20 tokens with their character offsets:
  Token                Char offset
  -----------------------------------
  [CLS]                (0, 0)
  what                 (0, 4)
  percentage           (5, 15)
  of                   (16, 18)
  egyptians            (19, 28)
  polled               (29, 35)
  support              (36, 43)
  death                (44, 49)
  penalty              (50, 57)
  for                  (58, 61)
  those                (62, 67)
  leaving              (68, 75)
  islam                (76, 81)
  ?                    (81, 82)
  [SEP]                (0, 0)
  the                  (0, 3)
  pew                  (4, 7)
  forum                (8, 13)
  on                   (14, 16)
  religion             (17, 25)


In [13]:
# ── Training tokenization function ───────────────────────────────────────────
def preprocess_training_examples(examples):
    """
    Tokenize training examples and compute token-level start/end positions.

    Key steps:
    1. Tokenize question + context with sliding window
    2. Use offset_mapping to find which tokens contain the answer
    3. If the answer is not in a chunk, label it (0, 0) — the [CLS] token
    4. If impossible to find exact token span, label it (0, 0)
    """
    # Strip leading/trailing whitespace from questions
    questions = [q.strip() for q in examples['question']]

    tokenized_examples = tokenizer(
        questions,
        examples['context'],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    # overflow_to_sample_mapping: maps each chunk back to its original example index
    # This is needed because one example can produce multiple chunks
    sample_mapping  = tokenized_examples.pop('overflow_to_sample_mapping')
    offset_mapping  = tokenized_examples.pop('offset_mapping')

    start_positions = []
    end_positions   = []

    for i, offsets in enumerate(offset_mapping):
        input_ids     = tokenized_examples['input_ids'][i]
        cls_index     = input_ids.index(tokenizer.cls_token_id)  # Always 0 for BERT

        # sequence_ids: 0=question tokens, 1=context tokens, None=special tokens
        sequence_ids  = tokenized_examples.sequence_ids(i)

        # Map this chunk back to its original example
        sample_index  = sample_mapping[i]
        answers       = examples['answers'][sample_index]

        # If no answers (impossible in SQuAD v1, possible in v2), default to CLS
        if len(answers['answer_start']) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Character-level start and end of the answer
        answer_start_char = answers['answer_start'][0]
        answer_end_char   = answer_start_char + len(answers['text'][0])

        # Find the start and end of the context in this chunk
        # (token index where context begins / ends)
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Check if the answer is completely inside this chunk
        # If not, label it with cls_index (will be ignored during evaluation)
        if not (
            offsets[token_start_index][0] <= answer_start_char
            and offsets[token_end_index][1] >= answer_end_char
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Move token_start_index to the first token whose offset >= answer_start_char
        while (
            token_start_index <= token_end_index
            and offsets[token_start_index][0] <= answer_start_char
        ):
            token_start_index += 1
        start_positions.append(token_start_index - 1)

        # Move token_end_index to the last token whose offset <= answer_end_char
        while (
            token_end_index >= token_start_index
            and offsets[token_end_index][1] >= answer_end_char
        ):
            token_end_index -= 1
        end_positions.append(token_end_index + 1)

    tokenized_examples['start_positions'] = start_positions
    tokenized_examples['end_positions']   = end_positions

    return tokenized_examples


print("Training tokenization function defined.")

Training tokenization function defined.


In [14]:
# ── Validation tokenization function ─────────────────────────────────────────
# Validation is different from training:
# We keep offset_mapping and example_id so we can reconstruct the answer text
# from token positions back to character positions after prediction.

def preprocess_validation_examples(examples):
    """
    Tokenize validation examples.
    We keep offset_mapping (for answer extraction) and
    example_id (to match predictions back to original examples).
    """
    questions = [q.strip() for q in examples['question']]

    tokenized_examples = tokenizer(
        questions,
        examples['context'],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized_examples.pop('overflow_to_sample_mapping')

    # Store the example_id for each feature so we can map back after prediction
    tokenized_examples['example_id'] = []

    for i in range(len(tokenized_examples['input_ids'])):
        sample_index = sample_mapping[i]
        tokenized_examples['example_id'].append(examples['id'][sample_index])

        # Set offset_mapping to None for question tokens and special tokens
        # so the answer extraction function knows to skip them
        sequence_ids = tokenized_examples.sequence_ids(i)
        tokenized_examples['offset_mapping'][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized_examples['offset_mapping'][i])
        ]

    return tokenized_examples


print("Validation tokenization function defined.")

Validation tokenization function defined.


In [15]:
# Apply tokenization to training set
print("Tokenizing training set...")
tokenized_train = train_data.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=train_data.column_names
)

# Apply tokenization to validation set
# We keep offset_mapping and example_id for post-processing
print("Tokenizing validation set...")
tokenized_eval = eval_data.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=eval_data.column_names
)

print(f"\nTokenized train size : {len(tokenized_train)}")
print(f"Tokenized eval size  : {len(tokenized_eval)}")
print()
print("Note: eval size > EVAL_SUBSET because long contexts produce multiple chunks.")
print(f"Sample keys (train) : {tokenized_train.column_names}")
print(f"Sample keys (eval)  : {tokenized_eval.column_names}")

Tokenizing training set...


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing validation set...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Tokenized train size : 8080
Tokenized eval size  : 1018

Note: eval size > EVAL_SUBSET because long contexts produce multiple chunks.
Sample keys (train) : ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']
Sample keys (eval)  : ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id']


In [17]:
# Verify one tokenized training example
sample_ex    = train_data[0]
sample_tok   = tokenized_train[0]

start_pos    = sample_tok['start_positions']
end_pos      = sample_tok['end_positions']
tokens       = tokenizer.convert_ids_to_tokens(sample_tok['input_ids'])

predicted_answer_tokens = tokens[start_pos : end_pos + 1]
predicted_answer_text   = tokenizer.convert_tokens_to_string(predicted_answer_tokens)

print("TRAINING LABEL VERIFICATION")
print(f"  Question         : {sample_ex['question']}")
print(f"  Gold answer      : '{sample_ex['answers']['text'][0]}'")
print(f"  start_positions  : {start_pos}")
print(f"  end_positions    : {end_pos}")
print(f"  Tokens at span   : {predicted_answer_tokens}")
print(f"  Decoded answer   : '{predicted_answer_text}'")

TRAINING LABEL VERIFICATION
  Question         : What percentage of Egyptians polled support death penalty for those leaving Islam?
  Gold answer      : '84%'
  start_positions  : 97
  end_positions    : 98
  Tokens at span   : ['84', '%']
  Decoded answer   : '84 %'


---
## Step 6 — Load BERT with QA Head

`AutoModelForQuestionAnswering` attaches a linear layer that produces
**two logit scores per token** — one for being the answer start,
one for being the answer end.

In [18]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

print(f"Model          : {MODEL_NAME}")
print(f"QA head        : {model.qa_outputs}")
print()
# The QA head: Linear(768, 2)
# Output: [batch, seq_len, 2] — two scores (start, end) per token

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Model          : bert-base-uncased
QA head        : Linear(in_features=768, out_features=2, bias=True)

Total parameters     : 108,893,186
Trainable parameters : 108,893,186


---
## Step 7 — Define Training Arguments

In [26]:
training_args = TrainingArguments(

    # ── Output & Logging ──────────────────────────────────────────────────
    output_dir=SAVE_DIR,
    logging_dir=f"{SAVE_DIR}/logs",
    logging_steps=100,
    report_to="none",

    # ── Training schedule ─────────────────────────────────────────────────
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",

    # ── Evaluation & Checkpointing ─────────────────────────────────────
    eval_strategy="no", # Changed from "epoch" to "no" as evaluation is done manually
    save_strategy="epoch",
    load_best_model_at_end=False, # Changed from True to False as eval_loss is not computed by Trainer
    # QA uses a custom compute_metrics so we skip metric_for_best_model here

    # ── Performance ───────────────────────────────────────────────────────
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

steps_per_epoch = len(tokenized_train) // BATCH_SIZE
total_steps     = steps_per_epoch * EPOCHS
print(f"Tokenized train samples : {len(tokenized_train)}")
print(f"Steps per epoch         : {steps_per_epoch}")
print(f"Total training steps    : {total_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Tokenized train samples : 8080
Steps per epoch         : 505
Total training steps    : 1010


---
## Step 8 — Define QA Evaluation Metrics

QA evaluation is unique — we cannot just compare predicted token indices.
We must:
1. Convert predicted start/end token indices → character spans → answer text string
2. Compare the predicted text against all gold answers

Two standard metrics:
- **Exact Match (EM)** — 1 if prediction exactly equals any gold answer, else 0
- **F1** — token-level overlap between prediction and gold answer

Example:
```
Gold   : "Paris, France"
Pred 1 : "Paris, France"   → EM=1.0, F1=1.0
Pred 2 : "Paris"           → EM=0.0, F1=0.67  (partial credit)
Pred 3 : "London"          → EM=0.0, F1=0.0
```

Because evaluation requires mapping predicted token positions back to
original text spans, we implement a custom `compute_metrics` function
that works outside the Trainer loop.

In [27]:
# Load the SQuAD metric from evaluate library
squad_metric = evaluate.load("squad")

def postprocess_qa_predictions(
    examples,
    features,
    raw_predictions,
    n_best_size=20,
    max_answer_length=30
):
    """
    Convert raw start/end logits from the model into answer text strings.

    Process:
    1. For each example, gather all its features (chunks from sliding window)
    2. For each feature, find the n_best start/end token combinations
    3. Convert token indices to character offsets using offset_mapping
    4. Extract the actual answer string from the original context
    5. Pick the best answer across all chunks
    """
    all_start_logits, all_end_logits = raw_predictions

    # Build a mapping: example_id → list of feature indices
    example_id_to_index = {ex['id']: i for i, ex in enumerate(examples)}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature['example_id']]].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        context         = example['context']
        valid_answers   = []

        for feature_index in feature_indices:
            start_logits   = all_start_logits[feature_index]
            end_logits     = all_end_logits[feature_index]
            offset_mapping = features[feature_index]['offset_mapping']

            # Find n_best start and end positions
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes   = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip invalid spans
                    if start_index >= len(offset_mapping):
                        continue
                    if end_index >= len(offset_mapping):
                        continue
                    if offset_mapping[start_index] is None:
                        continue
                    if offset_mapping[end_index] is None:
                        continue
                    if end_index < start_index:
                        continue
                    if end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char   = offset_mapping[end_index][1]
                    valid_answers.append({
                        'score': start_logits[start_index] + end_logits[end_index],
                        'text' : context[start_char:end_char]
                    })

        if valid_answers:
            best_answer = sorted(valid_answers, key=lambda x: x['score'], reverse=True)[0]
        else:
            best_answer = {'text': '', 'score': 0.0}

        predictions[example['id']] = best_answer['text']

    return predictions


print("QA postprocessing function defined.")
print("SQuAD metric loaded (Exact Match + F1).")

QA postprocessing function defined.
SQuAD metric loaded (Exact Match + F1).


---
## Step 9 — Train the Model

**What is different from Classification and NER training?**
- The model receives `start_positions` and `end_positions` as labels
- Two separate cross-entropy losses are computed (one for start, one for end)
- The total loss is their average
- We use `DefaultDataCollator` — no special padding needed since we padded to `max_length`
- Evaluation uses a custom postprocessing step (token indices → text strings)

In [28]:
# For QA we use DefaultDataCollator since we already padded to max_length
data_collator = DefaultDataCollator()

# We prepare a version of the eval dataset without example_id and offset_mapping
# because the Trainer cannot handle non-tensor columns during training loop
eval_dataset_for_trainer = tokenized_eval.remove_columns(['example_id', 'offset_mapping'])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=eval_dataset_for_trainer,
    data_collator=data_collator,
    # Note: compute_metrics is not passed here because QA metrics require
    # post-processing (token indices → text) which needs the original dataset.
    # We run evaluation manually after training.
)

print("Trainer built successfully.")
print(f"Training samples (including sliding window chunks) : {len(tokenized_train)}")

Trainer built successfully.
Training samples (including sliding window chunks) : 8080


In [29]:
# ── START TRAINING ──────────────────────────────────────────────────────────
# Expected time: 10-20 minutes on T4 GPU
# Expected EM: ~55-65, F1: ~65-75 on this subset

train_result = trainer.train()

print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(f"Training time         : {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples per second    : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"Final training loss   : {train_result.metrics['train_loss']:.4f}")

Step,Training Loss
100,1.543410
200,0.974945
300,0.673837
400,0.703604
500,0.720722
600,0.888167
700,0.853529
800,0.794289
900,0.816752
1000,0.785582


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


TRAINING COMPLETE
Training time         : 345s
Samples per second    : 46.8
Final training loss   : 0.8730


---
## Step 10 — Evaluate with Exact Match & F1

We run prediction on the validation set, then postprocess the raw logits
into answer strings and compare against gold answers.

In [30]:
# Get raw predictions (start_logits, end_logits) for all validation features
print("Running prediction on validation set...")
raw_predictions = trainer.predict(eval_dataset_for_trainer)

# raw_predictions.predictions is a tuple: (start_logits, end_logits)
# Each has shape [num_features, seq_len]
print(f"start_logits shape : {raw_predictions.predictions[0].shape}")
print(f"end_logits shape   : {raw_predictions.predictions[1].shape}")

Running prediction on validation set...


start_logits shape : (1018, 384)
end_logits shape   : (1018, 384)


In [31]:
# Postprocess: convert token logits → answer text strings
print("Postprocessing predictions → answer text...")
final_predictions = postprocess_qa_predictions(
    examples  = eval_data,
    features  = tokenized_eval,
    raw_predictions = raw_predictions.predictions
)

# Format for squad metric:
# predictions: list of {'id': str, 'prediction_text': str}
# references : list of {'id': str, 'answers': {'text': [...], 'answer_start': [...]}}
formatted_predictions = [
    {'id': k, 'prediction_text': v}
    for k, v in final_predictions.items()
]
formatted_references = [
    {'id': ex['id'], 'answers': ex['answers']}
    for ex in eval_data
]

results = squad_metric.compute(
    predictions=formatted_predictions,
    references=formatted_references
)

print("\n" + "=" * 50)
print("EVALUATION RESULTS (SQuAD metrics)")
print("=" * 50)
print(f"  Exact Match (EM) : {results['exact_match']:.2f}")
print(f"  F1 Score         : {results['f1']:.2f}")
print()
print("Interpretation:")
print(f"  EM={results['exact_match']:.1f} means the model's answer exactly matches")
print(f"  the gold answer for {results['exact_match']:.1f}% of questions.")
print(f"  F1={results['f1']:.1f} measures partial token-level overlap.")

Postprocessing predictions → answer text...

EVALUATION RESULTS (SQuAD metrics)
  Exact Match (EM) : 61.20
  F1 Score         : 76.25

Interpretation:
  EM=61.2 means the model's answer exactly matches
  the gold answer for 61.2% of questions.
  F1=76.2 measures partial token-level overlap.


In [32]:
# Show sample predictions vs gold answers
print("=" * 70)
print("SAMPLE PREDICTIONS vs GOLD ANSWERS")
print("=" * 70)

shown = 0
for ex in eval_data:
    if shown >= 8:
        break
    ex_id      = ex['id']
    gold       = ex['answers']['text'][0]
    prediction = final_predictions.get(ex_id, "")
    match      = "EXACT" if prediction.strip().lower() == gold.strip().lower() else "PARTIAL/WRONG"

    print(f"\n  Question   : {ex['question']}")
    print(f"  Gold       : '{gold}'")
    print(f"  Prediction : '{prediction}'")
    print(f"  Result     : {match}")
    shown += 1

SAMPLE PREDICTIONS vs GOLD ANSWERS

  Question   : In what year did Massachusetts first require children to be educated in schools?
  Gold       : '1852'
  Prediction : '1852.'
  Result     : PARTIAL/WRONG

  Question   : When were stromules discovered?
  Gold       : '1962'
  Prediction : '1962,'
  Result     : PARTIAL/WRONG

  Question   : Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
  Gold       : 'Horace Walpole'
  Prediction : 'Horace Walpole'
  Result     : EXACT

  Question   : In 1890, who did the university decide to team up with?
  Gold       : 'several regional colleges and universities'
  Prediction : 'the University of Chicago'
  Result     : PARTIAL/WRONG

  Question   : Who got a touchdown making the score 10-7?
  Gold       : 'Jonathan Stewart'
  Prediction : 'Jonathan Stewart'
  Result     : EXACT

  Question   : How many Examination Boards exist in India?
  Gold       : '30'
  Prediction : '30 different'
 

---
## Step 11 — Run Inference on Custom Questions

Test your model on entirely new question-context pairs.

In [33]:
# Manual inference — see every step
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def answer_question(question, context):
    """
    Manually extract an answer span from the context.
    Shows every step: tokenize → forward → decode.
    """
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=MAX_LENGTH,
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop('offset_mapping')[0].tolist()
    sequence_ids   = inputs.sequence_ids(0)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0].cpu().numpy()
    end_logits   = outputs.end_logits[0].cpu().numpy()

    # Find best valid start/end pair
    best_score  = float('-inf')
    best_start  = 0
    best_end    = 0

    for start_idx in np.argsort(start_logits)[-20:][::-1]:
        for end_idx in np.argsort(end_logits)[-20:][::-1]:
            if sequence_ids[start_idx] != 1:
                continue
            if sequence_ids[end_idx] != 1:
                continue
            if end_idx < start_idx:
                continue
            if end_idx - start_idx + 1 > 30:
                continue
            score = start_logits[start_idx] + end_logits[end_idx]
            if score > best_score:
                best_score = score
                best_start = start_idx
                best_end   = end_idx

    start_char = offset_mapping[best_start][0]
    end_char   = offset_mapping[best_end][1]
    answer     = context[start_char:end_char]

    return answer


# Custom test examples
qa_examples = [
    {
        "question": "What is the capital of France?",
        "context" : "France is a country in Western Europe. Its capital city is Paris, which is known for the Eiffel Tower and world-class cuisine."
    },
    {
        "question": "Who founded Microsoft?",
        "context" : "Microsoft Corporation was founded by Bill Gates and Paul Allen on April 4, 1975, in Albuquerque, New Mexico."
    },
    {
        "question": "In which city is the Taj Mahal located?",
        "context" : "The Taj Mahal is an ivory-white marble mausoleum located in Agra, Uttar Pradesh, India. It was commissioned by Mughal Emperor Shah Jahan."
    },
    {
        "question": "What is the speed of light?",
        "context" : "Light travels at approximately 299,792,458 metres per second in a vacuum. This constant is denoted as 'c' in physics equations."
    },
    {
        "question": "When was BERT published?",
        "context" : "BERT, which stands for Bidirectional Encoder Representations from Transformers, was introduced by researchers at Google AI Language in 2018."
    },
]

print("=" * 65)
print("MANUAL INFERENCE ON CUSTOM QUESTIONS")
print("=" * 65)

for example in qa_examples:
    answer = answer_question(example['question'], example['context'])
    print(f"\n  Question : {example['question']}")
    print(f"  Answer   : '{answer}'")

MANUAL INFERENCE ON CUSTOM QUESTIONS

  Question : What is the capital of France?
  Answer   : 'Paris,'

  Question : Who founded Microsoft?
  Answer   : 'Bill Gates and Paul Allen'

  Question : In which city is the Taj Mahal located?
  Answer   : 'Agra, Uttar Pradesh'

  Question : What is the speed of light?
  Answer   : '299,792,458 metres per second'

  Question : When was BERT published?
  Answer   : '2018.'


In [34]:
# Using Hugging Face pipeline — much simpler for inference
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

print("=" * 65)
print("PIPELINE INFERENCE")
print("=" * 65)

for example in qa_examples:
    result = qa_pipeline(
        question=example['question'],
        context=example['context']
    )
    print(f"\n  Question : {example['question']}")
    print(f"  Answer   : '{result['answer']}' (score: {result['score']:.4f})")

PIPELINE INFERENCE

  Question : What is the capital of France?
  Answer   : 'Paris,' (score: 0.7351)

  Question : Who founded Microsoft?
  Answer   : 'Bill Gates and Paul Allen' (score: 0.8923)

  Question : In which city is the Taj Mahal located?
  Answer   : 'Agra, Uttar Pradesh' (score: 0.5917)

  Question : What is the speed of light?
  Answer   : '299,792,458 metres per second' (score: 0.2407)

  Question : When was BERT published?
  Answer   : '2018.' (score: 0.9214)


---
## Step 12 — Save the Model Locally

In [35]:
import os

FINAL_SAVE_DIR = "./bert-squad-qa-final"
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

model.save_pretrained(FINAL_SAVE_DIR)
tokenizer.save_pretrained(FINAL_SAVE_DIR)

print(f"Model saved to: {FINAL_SAVE_DIR}")
print("\nSaved files:")
for f in sorted(os.listdir(FINAL_SAVE_DIR)):
    size = os.path.getsize(f"{FINAL_SAVE_DIR}/{f}")
    print(f"  {f:<45s} {size/1e6:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./bert-squad-qa-final

Saved files:
  config.json                                   0.0 MB
  model.safetensors                             435.6 MB
  tokenizer.json                                0.7 MB
  tokenizer_config.json                         0.0 MB


In [36]:
# Verify local save
print("Verifying saved model...")
reloaded_model     = AutoModelForQuestionAnswering.from_pretrained(FINAL_SAVE_DIR)
reloaded_tokenizer = AutoTokenizer.from_pretrained(FINAL_SAVE_DIR)

verify_pipeline = pipeline(
    "question-answering",
    model=reloaded_model,
    tokenizer=reloaded_tokenizer
)

test_result = verify_pipeline(
    question="What did BERT stand for?",
    context="BERT stands for Bidirectional Encoder Representations from Transformers and was released in 2018."
)
print(f"Answer : '{test_result['answer']}'")
print("Local save verified!")

Verifying saved model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Answer : 'Bidirectional Encoder Representations from Transformers'
Local save verified!


---
## Step 13 — Write Model Card & Push to Hugging Face Hub

In [37]:
em_score = results['exact_match']
f1_score = results['f1']

model_card = f"""---
language: en
license: apache-2.0
base_model: bert-base-uncased
tags:
  - question-answering
  - bert
  - squad
  - extractive-qa
datasets:
  - rajpurkar/squad
metrics:
  - exact_match
  - f1
---

# BERT SQuAD Question Answering Model

A fine-tuned version of `bert-base-uncased` on [SQuAD v1.1](https://huggingface.co/datasets/rajpurkar/squad)
for **extractive question answering**.

This model finds answer spans directly within a provided context paragraph.
It does not generate new text — the answer must exist in the context.

## Model Performance
Evaluated on {EVAL_SUBSET} examples from the SQuAD v1.1 validation set:

| Metric | Score |
|---|---|
| Exact Match (EM) | {em_score:.2f} |
| F1 Score | {f1_score:.2f} |

## How to Use

```python
from transformers import pipeline

qa = pipeline("question-answering", model="{HF_REPO_ID}")

result = qa(
    question="What is the capital of France?",
    context="France is a country in Western Europe. Its capital city is Paris."
)
print(result)
# {{'answer': 'Paris', 'score': 0.98, 'start': 58, 'end': 63}}
```

## Input Format
- **question**: The question to answer (string)
- **context**: The paragraph containing the answer (string)
- The answer must exist verbatim within the context
- Max combined input length: {MAX_LENGTH} tokens
- Longer contexts are handled automatically via sliding window (stride={DOC_STRIDE})

## Training Details
| Parameter | Value |
|---|---|
| Base model | bert-base-uncased |
| Dataset | rajpurkar/squad (v1.1) |
| Training samples | {TRAIN_SUBSET} |
| Epochs | {EPOCHS} |
| Batch size | {BATCH_SIZE} |
| Learning rate | {LEARNING_RATE} |
| Max length | {MAX_LENGTH} |
| Doc stride | {DOC_STRIDE} |
| Warmup ratio | {WARMUP_RATIO} |
| Optimizer | AdamW with linear LR decay |
"""

with open(f"{FINAL_SAVE_DIR}/README.md", "w") as f:
    f.write(model_card)

print("Model card written.")

Model card written.


In [38]:
# Create Hub repository and push all files
api = HfApi()

try:
    api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
    print(f"Repository ready: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Repo note: {e}")

print("\nPushing model...")
model.push_to_hub(
    HF_REPO_ID,
    commit_message="Add fine-tuned BERT SQuAD QA model"
)

print("Pushing tokenizer...")
tokenizer.push_to_hub(
    HF_REPO_ID,
    commit_message="Add tokenizer"
)

print("Pushing model card...")
api.upload_file(
    path_or_fileobj=f"{FINAL_SAVE_DIR}/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    commit_message="Add model card"
)

print("\n" + "=" * 55)
print("MODEL PUBLISHED SUCCESSFULLY")
print("=" * 55)
print(f"View at : https://huggingface.co/{HF_REPO_ID}")

Repository ready: https://huggingface.co/argha9177/bert-squad-qa

Pushing model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...yebdbov/model.safetensors:   0%|          | 14.2kB /  436MB            

Pushing tokenizer...


README.md: 0.00B [00:00, ?B/s]

Pushing model card...

MODEL PUBLISHED SUCCESSFULLY
View at : https://huggingface.co/argha9177/bert-squad-qa


---
## Step 14 — Load and Test the Published Model

In [39]:
# Load directly from Hugging Face Hub
print(f"Loading from Hub: {HF_REPO_ID}")

published_qa = pipeline(
    "question-answering",
    model=HF_REPO_ID
)

final_tests = [
    {
        "question": "Who invented the telephone?",
        "context" : "The telephone was invented by Alexander Graham Bell, a Scottish-American inventor, in 1876. Bell was awarded the first patent for the telephone."
    },
    {
        "question": "Which planet is closest to the Sun?",
        "context" : "The solar system consists of eight planets. Mercury is the smallest planet and the closest to the Sun, orbiting at an average distance of 57.9 million kilometres."
    },
    {
        "question": "What language is used to train most NLP models?",
        "context" : "Python has become the dominant programming language in machine learning and NLP research. Libraries like PyTorch, TensorFlow, and Hugging Face Transformers are all Python-based."
    },
]

print("\n" + "=" * 65)
print("FINAL INFERENCE FROM PUBLISHED HUB MODEL")
print("=" * 65)

for test in final_tests:
    result = published_qa(
        question=test['question'],
        context=test['context']
    )
    print(f"\n  Question : {test['question']}")
    print(f"  Answer   : '{result['answer']}' (score: {result['score']:.4f})")

Loading from Hub: argha9177/bert-squad-qa


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


FINAL INFERENCE FROM PUBLISHED HUB MODEL

  Question : Who invented the telephone?
  Answer   : 'Alexander Graham Bell' (score: 0.9493)

  Question : Which planet is closest to the Sun?
  Answer   : 'Mercury is' (score: 0.9954)

  Question : What language is used to train most NLP models?
  Answer   : 'Python has' (score: 0.9532)


---
## Summary — Complete Comparison of All Three Tasks

```
TEXT CLASSIFICATION      NER                      QUESTION ANSWERING
────────────────────     ──────────────────────   ──────────────────────────────
Input:
  "I love this film"     ["I","love","Paris"]     question + context paragraph

BERT input format:
  [CLS] text [SEP]       [CLS] tokens [SEP]       [CLS] Q [SEP] Context [SEP]

BERT output used:
  [CLS] vector only      ALL token vectors        ALL token vectors

Head (Linear layer):
  768 → num_labels       768 → num_labels          768 → 2  (start + end)
  (one output)           (per token)               (per token)

Labels:
  [1]                    [-100,0,0,3,4,0,-100]    start_pos=4, end_pos=6

Key challenge:
  none special           subword alignment         offset mapping + sliding window

Data collator:
  default                DataCollatorForToken      DefaultDataCollator
                         Classification

Metric:
  accuracy / F1          seqeval F1 (entity)       Exact Match + F1 (token)

Model class:
  AutoModelFor           AutoModelFor              AutoModelFor
  SequenceClassification TokenClassification       QuestionAnswering
```

---

## Key Concepts Reinforced

| Concept | Where you used it |
|---|---|
| `return_offsets_mapping=True` | Maps tokens back to character positions |
| `return_overflowing_tokens=True` | Creates sliding window chunks for long contexts |
| `truncation="only_second"` | Never truncates the question, only context |
| `overflow_to_sample_mapping` | Maps each chunk back to its original example |
| `sequence_ids()` | Identifies question vs context tokens in each chunk |
| `start_positions` / `end_positions` | Token index labels for QA training |
| `postprocess_qa_predictions()` | Converts logits → answer text after prediction |
| Exact Match + F1 | QA-specific metrics via `evaluate.load('squad')` |

---

## Huggingface Repo

https://huggingface.co/argha9177/bert-squad-qa